# 🩺 Diabetes Disease Prediction - Model Training
## Explainable AI (XAI) Based Multi-Disease Risk Prediction System
### Dataset: Pima Indians Diabetes Dataset (768 samples, 8 features)

**Models Trained:** Logistic Regression, Random Forest, XGBoost, Decision Tree  
**XAI Method:** SHAP (SHapley Additive exPlanations)  
**Output:** Best model saved as `diabetes_model.pkl`

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
import shap

# Output directory for saved models
os.makedirs('../backend/models', exist_ok=True)
print('✅ Libraries loaded successfully')

## 1️⃣ Data Loading & Exploratory Data Analysis

In [ ]:
# Load dataset
df = pd.read_csv('diabetes.csv')  # Place dataset in notebooks/ folder
print(f'Dataset Shape: {df.shape}')
df.head(10)

In [ ]:
print('📊 Dataset Info:')
print(df.info())
print('\n📈 Statistical Summary:')
df.describe().T

In [ ]:
print('🎯 Target Distribution:')
print(df['Outcome'].value_counts())
print(f'Class Balance: {df["Outcome"].value_counts(normalize=True).round(3) * 100}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Outcome'].value_counts().plot(kind='bar', ax=axes[0], color=['#00b4d8','#e63946'], edgecolor='black')
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Outcome (0=No Diabetes, 1=Diabetes)')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

df['Outcome'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                   colors=['#00b4d8','#e63946'], startangle=90)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Distributions
features = df.columns[:-1].tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df[df['Outcome']==0][col], alpha=0.6, label='No Diabetes', color='#00b4d8', bins=25)
    axes[i].hist(df[df['Outcome']==1][col], alpha=0.6, label='Diabetes', color='#e63946', bins=25)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)
plt.suptitle('Feature Distributions by Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_feat_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_corr.png', dpi=150, bbox_inches='tight')
plt.show()

## 2️⃣ Data Preprocessing

In [ ]:
# Replace biologically impossible 0s with NaN (except Pregnancies & Outcome)
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)
print('Missing values after replacing zeros:')
print(df.isnull().sum())

# Fill with median grouped by Outcome
for col in zero_cols:
    df[col] = df.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

print('\n✅ Missing values handled')
print(df.isnull().sum())

In [ ]:
# Feature Engineering
df['BMI_Age'] = df['BMI'] * df['Age']
df['Glucose_Insulin'] = df['Glucose'] / (df['Insulin'] + 1)
df['Glucose_BMI'] = df['Glucose'] * df['BMI']

FEATURE_NAMES = [
    'Pregnancies','Glucose','BloodPressure','SkinThickness',
    'Insulin','BMI','DiabetesPedigreeFunction','Age',
    'BMI_Age','Glucose_Insulin','Glucose_BMI'
]

X = df[FEATURE_NAMES]
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

## 3️⃣ Model Training & Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, min_samples_split=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5,
                                         use_label_encoder=False, eval_metric='logloss', random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    X_tr = X_train_sc if name in ['Logistic Regression'] else X_train
    X_te = X_test_sc  if name in ['Logistic Regression'] else X_test
    X_cv = X_train_sc if name in ['Logistic Regression'] else X_train

    cv_scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    trained_models[name] = {'model': model, 'X_test': X_te, 'X_train': X_tr}
    results.append({'Model': name, 'CV Accuracy': cv_scores.mean(),
                    'Test Accuracy': acc, 'AUC-ROC': auc,
                    'F1 Score': f1, 'Precision': prec, 'Recall': rec})

    print(f'✅ {name}: Acc={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f} | CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}')

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
results_df

In [ ]:
# Model Comparison Plot
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
w = 0.15
metrics = ['CV Accuracy', 'Test Accuracy', 'AUC-ROC', 'F1 Score']
colors  = ['#00b4d8', '#0077b6', '#e63946', '#2ec4b6']
for i, (m, c) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*w, results_df[m], w, label=m, color=c, edgecolor='white')
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(results_df['Model'], fontsize=10)
ax.set_ylim(0.5, 1.0)
ax.set_title('🔬 Model Performance Comparison - Diabetes', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/models/diabetes_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
for name, data in trained_models.items():
    y_prob = data['model'].predict_proba(data['X_test'])[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
plt.plot([0,1],[0,1],'k--', label='Random', linewidth=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Diabetes Models', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/models/diabetes_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, data) in zip(axes, trained_models.items()):
    y_pred = data['model'].predict(data['X_test'])
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['No DM','DM'], yticklabels=['No DM','DM'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices - Diabetes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4️⃣ SHAP Explainability (Best Model)

In [ ]:
# Best model = XGBoost (typically highest AUC)
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]['model']
X_test_best = trained_models[best_name]['X_test']

print(f'🏆 Best Model: {best_name}')

# SHAP Explainer
if best_name == 'XGBoost':
    explainer = shap.TreeExplainer(best_model)
elif best_name == 'Random Forest':
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.LinearExplainer(best_model, X_train_sc)

shap_values = explainer.shap_values(X_test_best)
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# Global SHAP Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test_best,
                  feature_names=FEATURE_NAMES if best_name in ['XGBoost','Random Forest'] else FEATURE_NAMES,
                  show=False)
plt.title(f'SHAP Summary - {best_name} (Diabetes)', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (Global Feature Importance)
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_vals, X_test_best, feature_names=FEATURE_NAMES, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/diabetes_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Force Plot for a single prediction (example)
idx = 0
print(f'\n🔍 SHAP Explanation for Sample #{idx}')
print(f'True Label: {y_test.iloc[idx]}')
pred = best_model.predict(X_test_best[[idx]])[0]
prob = best_model.predict_proba(X_test_best[[idx]])[0][1]
print(f'Predicted: {pred}, Probability: {prob:.4f}')
shap.force_plot(explainer.expected_value if not isinstance(explainer.expected_value, list)
                else explainer.expected_value[1],
                shap_vals[idx], X_test_best[idx] if hasattr(X_test_best,'iloc') == False
                else X_test_best.iloc[idx].values,
                feature_names=FEATURE_NAMES, matplotlib=True, show=False)
plt.savefig('../backend/models/diabetes_shap_force.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Save Best Model & Artifacts

In [ ]:
# Save best model, scaler, feature names and model comparison
artifact = {
    'model': best_model,
    'model_name': best_name,
    'scaler': scaler,
    'feature_names': FEATURE_NAMES,
    'model_results': results_df.to_dict('records'),
    'disease': 'diabetes',
    'target_col': 'Outcome',
    'classes': ['No Diabetes', 'Diabetes'],
    'threshold': 0.5
}

with open('../backend/models/diabetes_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f'✅ Model saved → backend/models/diabetes_model.pkl')
print(f'🏆 Best Model: {best_name}')
print(f'📊 Test Accuracy: {results_df.iloc[0]["Test Accuracy"]:.4f}')
print(f'📊 AUC-ROC: {results_df.iloc[0]["AUC-ROC"]:.4f}')
print('\n📋 All Model Results:')
print(results_df.to_string(index=False))